In [1]:
from robot import SerialPrinter, Printer
from datatypes import RobotCalibration

In [2]:
printer = SerialPrinter()
printer.connect()

Connecting to GRBL on 0...
Connecting to GRBL on 1...
Connecting to GRBL on 2...
Connecting to GRBL on 3...
Connecting to GRBL on 4...
Connecting to GRBL on 5...
Waiting...
Connected successfully.
start
Marlin 2.1.2.5
echo: Last Updated: 2024-11-18 | Author: (none, default config)
echo: Compiled: Jun 10 2026
echo: Free Memory: 5619  PlannerBufferBytes: 1136
echo:Hardcoded Default Settings Loaded
echo:; Linear Units:
echo:  G21 ; (mm)
echo:; Temperature Units:
echo:  M149 C ; Units in Celsius
echo:; Filament settings (Disabled):
echo:  M200 S0 D1.75
echo:; Steps per unit:
echo:  M92 X80.00 Y80.00 Z400.00 E500.00
echo:; Max feedrates (units/s):
echo:  M203 X300.00 Y300.00 Z5.00 E25.00
echo:; Max Acceleration (units/s2):
echo:  M201 X3000.00 Y3000.00 Z100.00 E10000.00
echo:; Acceleration (units/s2) (P<print-accel> R<retract-accel> T<travel-accel>):
echo:  M204 P3000.00 R3000.00 T3000.00
echo:; Advanced (B<min_segment_time_us> S<min_feedrate> T<min_travel_feedrate> J<junc_dev>):
echo:  M20

## Load Printer Calibration

In [3]:
my_robot_calibration = RobotCalibration.load("my_robot_calibration.json")

In [8]:
import math

def load_brush(printer: Printer, my_robot_calibration, color_index):
    """
    1. Navigates to the water cup and agitates to wet the brush.
    2. Navigates to a color palette position and swirls to load paint.
    
    :param printer: An instance of the SerialPrinter/Printer class
    :param palette_position: A tuple of (X, Y, Z) for the specific paint well
    """
    Z_MARGIN = 5
    FEED_RATE_TRAVEL = 2000
    FEED_RATE_WET = 1000
    FEED_RATE_LOAD = 500

    x_paint, y_paint, z_paint = my_robot_calibration.color_palette.color_positions[color_index]["position"]
    x_water, y_water, z_water = my_robot_calibration.water_reservoir

    print("--- Starting Full Brush Prep Sequence ---")

    # ==========================================
    # PHASE 1: WET THE BRUSH IN WATER
    # ==========================================
    print(f"Moving to water reservoir at ({x_water}, {y_water})...")
    # Lift to safe height, move to water cup, and dip down
    printer.move_to(z=z_water+Z_MARGIN, feed_rate=FEED_RATE_TRAVEL)
    printer.move_to(x=x_water, y=y_water, feed_rate=FEED_RATE_TRAVEL)
    printer.move_to(z=z_water, feed_rate=FEED_RATE_TRAVEL)

    print("Agitating brush in water...")
    # Perform a rapid mechanical "shake" to flex bristles and soak up water
    shake_distance = 4.0  # mm
    for _ in range(3):
        # Shake left/right, up/down relative to the center of the water cup
        printer.move_to(x=x_water + shake_distance, y=y_water, feed_rate=FEED_RATE_WET)
        printer.move_to(x=x_water - shake_distance, y=y_water, feed_rate=FEED_RATE_WET)
        printer.move_to(x=x_water, y=y_water + shake_distance, feed_rate=FEED_RATE_WET)
        printer.move_to(x=x_water, y=y_water - shake_distance, feed_rate=FEED_RATE_WET)
    
    # Return cleanly to center of water cup and lift up
    printer.move_to(x=x_water, y=y_water, feed_rate=FEED_RATE_WET)
    printer.move_to(z=z_water+Z_MARGIN, feed_rate=FEED_RATE_TRAVEL)


    # ==========================================
    # PHASE 2: LOAD THE PAINT
    # ==========================================
    print(f"Moving to paint palette at ({x_paint}, {y_paint})...")
    # Move over the target well, dip down to paint height
    printer.move_to(x=x_paint, y=y_paint, feed_rate=FEED_RATE_TRAVEL)
    printer.move_to(z=z_paint, feed_rate=FEED_RATE_TRAVEL)

    print("Swirling brush to load paint...")
    circle_radius = 5.0  # mm
    steps = 16
    for i in range(steps + 1):
        angle = (2 * math.pi / steps) * i
        circle_x = x_paint + circle_radius * math.cos(angle)
        circle_y = y_paint + circle_radius * math.sin(angle)
        printer.move_to(x=circle_x, y=circle_y, feed_rate=FEED_RATE_LOAD)

    # Move back up to clear the well completely before drawing or traveling
    printer.move_to(z=z_paint+Z_MARGIN, feed_rate=FEED_RATE_TRAVEL)
    
    print("--- Brush prep complete and ready to paint! ---")

In [9]:
color_index = 0 # red
load_brush(printer, my_robot_calibration, color_index)

--- Starting Full Brush Prep Sequence ---
Moving to water reservoir at (186.6, 179.5)...
Sending absolute move command: G1 Z5.000 F2000
ok
Sending absolute move command: G1 X186.600 Y179.500 F2000
ok
Sending absolute move command: G1 Z0.000 F2000
ok
Agitating brush in water...
Sending absolute move command: G1 X190.600 Y179.500 F1000
ok
Sending absolute move command: G1 X182.600 Y179.500 F1000
ok
Sending absolute move command: G1 X186.600 Y183.500 F1000
ok
Sending absolute move command: G1 X186.600 Y175.500 F1000
ok
Sending absolute move command: G1 X190.600 Y179.500 F1000
ok
Sending absolute move command: G1 X182.600 Y179.500 F1000
ok
Sending absolute move command: G1 X186.600 Y183.500 F1000
ok
Sending absolute move command: G1 X186.600 Y175.500 F1000
ok
Sending absolute move command: G1 X190.600 Y179.500 F1000
ok
Sending absolute move command: G1 X182.600 Y179.500 F1000
ok
Sending absolute move command: G1 X186.600 Y183.500 F1000
ok
Sending absolute move command: G1 X186.600 Y175.500